In [15]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


In [16]:
df = pd.read_csv("clean.csv")

target = "Anxiety Level (1-10)"

X = df.drop(columns=[target])
y = df[target]


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [18]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])


In [19]:
pipelines = {
    "linear": Pipeline([
        ("preprocess", preprocess),
        ("model", LinearRegression())
    ]),

    "ridge": Pipeline([
        ("preprocess", preprocess),
        ("model", Ridge())
    ]),

    "tree": Pipeline([
        ("preprocess", preprocess),
        ("model", DecisionTreeRegressor(random_state=42))
    ]),

    "rf": Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestRegressor(random_state=42))
    ]),

    "gboost": Pipeline([
        ("preprocess", preprocess),
        ("model", GradientBoostingRegressor(random_state=42))
    ])
}


In [20]:
param_grids = {
    "linear": {},

    "ridge": {
        "model__alpha": [0.1, 1, 10]
    },

    "tree": {
        "model__max_depth": [3, 5, 10]
    },

    "rf": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10]
    },

    "gboost": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1]
    }
}


In [21]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = {}

for name in pipelines:
    print(f"\nTraining {name}...")

    grid = GridSearchCV(
        pipelines[name],
        param_grids[name],
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )

    # Train on training set only
    grid.fit(X_train, y_train)

    # ----- CV performance -----
    cv_rmse = np.sqrt(-grid.best_score_)

    # ----- Test performance (unseen data) -----
    best_model = grid.best_estimator_
    y_pred_test = best_model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

    results[name] = {
        "grid": grid,
        "cv_rmse": cv_rmse,
        "test_rmse": test_rmse
    }

    print(f"{name} CV RMSE: {cv_rmse:.3f}")
    print(f"{name} Test RMSE: {test_rmse:.3f}")


Training linear...
linear CV RMSE: 1.127
linear Test RMSE: 1.126

Training ridge...
ridge CV RMSE: 1.127
ridge Test RMSE: 1.126

Training tree...
tree CV RMSE: 1.076
tree Test RMSE: 1.059

Training rf...
rf CV RMSE: 1.028
rf Test RMSE: 1.006

Training gboost...
gboost CV RMSE: 1.030
gboost Test RMSE: 1.016


In [22]:
# Pick best model based on TEST RMSE
best_model_name = min(results, key=lambda x: results[x]["test_rmse"])
best_grid = results[best_model_name]["grid"]

print("\nBest model selected:", best_model_name)
print("Best Test RMSE:", results[best_model_name]["test_rmse"])




Best model selected: rf
Best Test RMSE: 1.00636894616284


In [23]:
print("\nTraining final model on full dataset...")

final_model = best_grid.best_estimator_
final_model.fit(X, y)

joblib.dump(final_model, "regression_model.pkl")
print("Final model saved.")



Training final model on full dataset...


Final model saved.
